## Load the dataset

This should already be filtered for high quality german sentences

In [1]:
import pandas as pd

dataset = pd.read_json("../data/interim/german_sentences_filtered.jsonl", lines=True)
dataset.head()

,original_sentence,cleaned_sentence,keep,reason
0,Alan Smithee steht als Pseudonym für einen fik...,Alan Smithee steht als Pseudonym für einen fik...,True,good example
1,Von 1968 bis 2000 wurde es von der Directors G...,Von 1968 bis 2000 wurde es von der Directors G...,True,proper structure
2,Alternative Schreibweisen sind unter anderem d...,Alternative Schreibweisen sind unter anderem d...,True,valid sentence
3,Alan Smi Thee und Sumishii Aran gehören so die...,Alan Smi Thee und Sumishii Aran gehören so die...,True,useful structure
4,Regisseur Robert Totten und Hauptdarsteller Ri...,Regisseur Robert Totten und Hauptdarsteller Ri...,True,clear example


# Create Ollama client and Load Env

Load `OLLAMA_API_KEY` variable

In [2]:
from ollama import Client
import os
from dotenv import load_dotenv

load_dotenv() 

client = Client(
            host="https://ollama.com",
            headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
        )

## Create corruption pair

- Using a bigger LLM `GPT-OSS-120b` we create a corrupted pair for each sentence
- The number of issues and the types are also recorded

In [3]:
import json
from src.utils import process_in_batches
from src.data.corruptor import corrupt_sentence
from tqdm import tqdm

all_sentences = dataset["cleaned_sentence"].to_list()[759:]

print(f"Starting to corrupt {len(all_sentences)} rows...")

with tqdm(total=len(all_sentences), desc="Corrupting sentences") as pbar:
    for current_batch in process_in_batches(all_sentences, batch_size=10, verbose=False):
        batch_result = corrupt_sentence(current_batch, client)

        with open("../data/interim/german_sentences_filtered_corrupted.jsonl", "a", encoding="utf-8") as f:
            for line in batch_result:
                f.write(json.dumps(line, ensure_ascii=False) + "\n")
        
        pbar.update(len(current_batch))


Starting to corrupt 2670 rows...


Corrupting sentences: 100%|██████████| 2670/2670 [2:49:55<00:00,  3.82s/it]  


## We filter the data once more for Quality

As the last step `GPT-OSS-120b` goes through the pairs again and removes examples that don't have enough value for training or invalid

In [4]:
df_corrupted = pd.read_json("../data/interim/german_sentences_filtered_corrupted.jsonl", lines=True)
candidates = list(zip(df_corrupted["original"], df_corrupted["corrupted"]))

In [5]:
from tqdm import tqdm
from src.data.quality import validate_corruption_pair
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 3


def process_pair(pair):
    clean, corrupt = pair
    evaluation = validate_corruption_pair(clean, corrupt, client)
    return { "input": corrupt, "label": clean, "verdict": evaluation["verdict"], "reason": evaluation["reason"] }


with tqdm(total=len(candidates), desc="Evaluating pairs") as pbar:
    for i in range(0, len(candidates), BATCH_SIZE):
        batch = candidates[i:i + BATCH_SIZE]
        with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
            futures = {executor.submit(process_pair, pair): pair for pair in batch}
            for future in as_completed(futures):
                evaluated_data = future.result()
                with open("../data/interim/german_sentences_filtered_corrupted_evaluated.jsonl", "a", encoding="utf-8") as f:
                    f.write(json.dumps(evaluated_data, ensure_ascii=False) + "\n")
                pbar.update(1)

Evaluating pairs: 100%|██████████| 3434/3434 [3:25:22<00:00,  3.59s/it]   


# Create the final dataset ready for training

In [7]:
df_eval = pd.read_json("../data/interim/german_sentences_filtered_corrupted_evaluated.jsonl", lines=True)

df_final = df_eval[df_eval["verdict"] == "KEEP"].copy()
df_final.rename(columns={"input": "corrupted", "label": "original"}, inplace=True)
df_final.to_json("../data/processed/german_sentences_corrupted_final.jsonl", orient="records", lines=True, force_ascii=False)